## Import thư viện và khai báo đường dẫn

In [8]:
import pandas as pd
from pathlib import Path

# Notebook đang nằm trong thư mục notebooks
ROOT_DIR = Path.cwd().parent

# Dataset nguồn
DATA_DIR = ROOT_DIR / "Dataset" / "Amazon E-commerce Products & Reviews Dataset"

# Input files
PRODUCTS_PATH = DATA_DIR / "products.csv"
REVIEWS_PATH = DATA_DIR / "reviews.csv"

# Output
OUTPUT_DIR = ROOT_DIR / "Dataset" / "raw"
OUTPUT_PATH = OUTPUT_DIR / "amazon_products_reviews_merged.csv"

print("Products:", PRODUCTS_PATH)
print("Reviews :", REVIEWS_PATH)
print("Products exists:", PRODUCTS_PATH.exists())
print("Reviews exists :", REVIEWS_PATH.exists())

Products: c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\Amazon E-commerce Products & Reviews Dataset\products.csv
Reviews : c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\Amazon E-commerce Products & Reviews Dataset\reviews.csv
Products exists: True
Reviews exists : True


## Đọc dữ liệu

Đọc 2 file dữ liệu gốc gồm:

- `products.csv`: chứa thông tin sản phẩm, seller, brand, giá, mô tả sản phẩm.
- `reviews.csv`: chứa thông tin đánh giá của khách hàng.

Sau khi đọc, in ra kích thước dữ liệu để kiểm tra số dòng và số cột ban đầu.

In [9]:
products = pd.read_csv(PRODUCTS_PATH)
reviews = pd.read_csv(REVIEWS_PATH)

print("Products shape:", products.shape)
print("Reviews shape:", reviews.shape)

Products shape: (728, 34)
Reviews shape: (6327, 23)


## Chuẩn hóa tên cột và khóa merge

Chuẩn hóa tên cột để tránh lỗi do khác biệt định dạng, ví dụ:

- Có khoảng trắng thừa.
- Có chữ hoa/chữ thường không thống nhất.
- Có ký tự `/`.
- Có khoảng trắng giữa các từ.

Sau đó kiểm tra 2 cột khóa bắt buộc:

- `asin` trong bảng products.
- `productasin` trong bảng reviews.

Cuối cùng, chuẩn hóa key merge bằng cách chuyển sang string, xóa khoảng trắng thừa và chuyển thành chữ hoa.

In [10]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df

products = normalize_columns(products)
reviews = normalize_columns(reviews)

if "asin" not in products.columns:
    raise KeyError("Thiếu cột 'asin' trong products.csv")

if "productasin" not in reviews.columns:
    raise KeyError("Thiếu cột 'productasin' trong reviews.csv")

products["asin"] = products["asin"].astype(str).str.strip().str.upper()
reviews["productasin"] = reviews["productasin"].astype(str).str.strip().str.upper()

## Kiểm tra mức độ khớp ASIN giữa hai bảng

Kiểm tra trước khi merge để biết dữ liệu có khớp nhau tốt không.

Các thông tin được in ra gồm:

- Số ASIN unique trong products.
- Số productASIN unique trong reviews.
- Số ASIN xuất hiện ở cả hai bảng.
- Tỷ lệ sản phẩm trong products có review tương ứng.
- Tỷ lệ review trong reviews có thông tin sản phẩm tương ứng.

In [11]:
product_asins = set(products["asin"].dropna().unique())
review_asins = set(reviews["productasin"].dropna().unique())

matched_asins = product_asins.intersection(review_asins)

print("Số ASIN unique ở products:", len(product_asins))
print("Số productASIN unique ở reviews:", len(review_asins))
print("Số ASIN match giữa 2 bảng:", len(matched_asins))

product_match_rate = len(matched_asins) / len(product_asins) if len(product_asins) > 0 else 0
review_match_rate = len(matched_asins) / len(review_asins) if len(review_asins) > 0 else 0

print(f"Tỷ lệ match của products: {product_match_rate:.2%}")
print(f"Tỷ lệ match của reviews: {review_match_rate:.2%}")

Số ASIN unique ở products: 728
Số productASIN unique ở reviews: 700
Số ASIN match giữa 2 bảng: 700
Tỷ lệ match của products: 96.15%
Tỷ lệ match của reviews: 100.00%


## Merge reviews với products

Merge bảng reviews với bảng products theo đúng khóa:

- `reviews.productasin`
- `products.asin`

Không dùng `s.no` để merge vì `s.no` chỉ là số thứ tự dòng, không đại diện cho sản phẩm.

Sử dụng `how="left"` để giữ toàn bộ review. Nếu review nào không tìm được thông tin product tương ứng thì các cột product sẽ bị missing.

Sau merge, kiểm tra:

- Kích thước dataset sau merge.
- Số dòng review không match được product info.
- Tỷ lệ thiếu `seller_name`.
- Top 20 cột có nhiều missing values nhất.

In [12]:
merged_df = reviews.merge(
    products,
    left_on="productasin",
    right_on="asin",
    how="left",
    suffixes=("_review", "_product")
)

print("Shape dataset sau merge:", merged_df.shape)

unmatched_reviews = merged_df["asin"].isna().sum()
print("Số dòng review không match được product info:", unmatched_reviews)

if "seller_name" in merged_df.columns:
    seller_missing_rate = merged_df["seller_name"].isna().mean()
    print(f"Tỷ lệ missing của seller_name: {seller_missing_rate:.2%}")
else:
    print("Cột seller_name không tồn tại sau merge.")

print("\nTop 20 missing values:")
display(
    merged_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Shape dataset sau merge: (6327, 57)
Số dòng review không match được product info: 0
Tỷ lệ missing của seller_name: 1.66%

Top 20 missing values:


images_7                   6322
images_6                   6320
default_variant_2          6317
images_5                   6313
images_4                   6301
videos_0                   6293
images_3                   6278
images_2                   6229
images_1                   6115
images_0                   5811
model_number               4564
product_description        3987
manufacturer               3968
list_price                 2711
seller_page_url            2652
rank_1                     1442
best_sellers_rank          1349
recent_purchases           1084
brand_page_url              591
customer_review_summary     512
dtype: int64

## Chọn cột quan trọng và tạo các trường text chuẩn hóa

Giữ lại các cột quan trọng phục vụ Seller ESG Scoring.

Các nhóm cột chính gồm:

- Thông tin sản phẩm.
- Thông tin seller.
- Thông tin brand/manufacturer.
- Thông tin review.
- Sentiment score nếu đã có.

Sau đó tạo thêm các cột chuẩn hóa:

- `product_text`: gộp title, about item và product description để phục vụ phân tích ESG từ nội dung sản phẩm.
- `review_text_clean`: ưu tiên dùng `cleaned_review_text`; nếu không có thì dùng `reviewtext`.
- `has_review`: đánh dấu dòng có review text hay không.
- `has_product_info`: đánh dấu dòng có thông tin seller/product hay không.

Lưu ý: không merge Amazon Fashion 800K vào dataset chính vì dataset đó không có `seller_name`. Dataset 800K nên để riêng cho NLP hoặc topic modeling.

In [13]:
important_columns = [
    "asin",
    "title_product",
    "about_item",
    "product_description",
    "availability",
    "brand_name",
    "manufacturer",
    "price_value",
    "rating_count",
    "rating_stars",
    "recent_purchases",
    "seller_name",
    "seller_page_url",
    "rank_1",
    "best_sellers_rank",

    "productasin",
    "productvariant",
    "rating",
    "reviewid",
    "reviewmetadata",
    "reviewtext",
    "reviewtitle",
    "verifiedpurchase",
    "helpfulvote",
    "cleaned_review_text",
    "sentiment_score"
]

existing_columns = [col for col in important_columns if col in merged_df.columns]
missing_columns = [col for col in important_columns if col not in merged_df.columns]

main_df = merged_df[existing_columns].copy()

print("Số cột được giữ:", len(existing_columns))
print("Các cột thiếu:", missing_columns)

def get_text_col(df, col_name):
    if col_name in df.columns:
        return df[col_name].fillna("").astype(str)
    return ""

main_df["product_text"] = (
    get_text_col(main_df, "title_product") + " " +
    get_text_col(main_df, "about_item") + " " +
    get_text_col(main_df, "product_description")
).str.strip()

if "cleaned_review_text" in main_df.columns and "reviewtext" in main_df.columns:
    main_df["review_text_clean"] = main_df["cleaned_review_text"].fillna(main_df["reviewtext"])
elif "cleaned_review_text" in main_df.columns:
    main_df["review_text_clean"] = main_df["cleaned_review_text"]
elif "reviewtext" in main_df.columns:
    main_df["review_text_clean"] = main_df["reviewtext"]
else:
    main_df["review_text_clean"] = None

if "reviewtext" in main_df.columns:
    main_df["has_review"] = main_df["reviewtext"].notna()
else:
    main_df["has_review"] = False

if "seller_name" in main_df.columns:
    main_df["has_product_info"] = main_df["seller_name"].notna()
else:
    main_df["has_product_info"] = False

print("Shape sau khi chọn cột và tạo text fields:", main_df.shape)

Số cột được giữ: 24
Các cột thiếu: ['title_product', 'helpfulvote']
Shape sau khi chọn cột và tạo text fields: (6327, 28)


## Xóa dữ liệu trùng lặp

Xóa duplicate để tránh một review bị tính nhiều lần trong quá trình chấm điểm ESG.

Ưu tiên xóa duplicate theo `reviewid` vì đây là mã định danh review rõ ràng nhất.

Nếu không có `reviewid`, dùng tổ hợp:

- `productasin`
- `reviewtext`
- `rating`

Nếu các cột này cũng không đầy đủ thì mới xóa duplicate toàn dòng.

In [14]:
before_dedup = main_df.shape[0]

if "reviewid" in main_df.columns:
    main_df = main_df.drop_duplicates(subset=["reviewid"])
    print("Đã xóa duplicate theo reviewid.")
else:
    duplicate_keys = [col for col in ["productasin", "reviewtext", "rating"] if col in main_df.columns]
    
    if len(duplicate_keys) > 0:
        main_df = main_df.drop_duplicates(subset=duplicate_keys)
        print(f"Đã xóa duplicate theo: {duplicate_keys}")
    else:
        main_df = main_df.drop_duplicates()
        print("Không có key phù hợp, đã xóa duplicate toàn dòng.")

after_dedup = main_df.shape[0]

print("Số dòng trước khi xóa duplicate:", before_dedup)
print("Số dòng sau khi xóa duplicate:", after_dedup)
print("Số dòng duplicate đã xóa:", before_dedup - after_dedup)

Đã xóa duplicate theo reviewid.
Số dòng trước khi xóa duplicate: 6327
Số dòng sau khi xóa duplicate: 6327
Số dòng duplicate đã xóa: 0


## Lưu dữ liệu và tổng kết cuối cùng

Tạo thư mục `data/processed` nếu chưa tồn tại, sau đó lưu dataset đã merge vào file:

`data/processed/amazon_products_reviews_merged.csv`

Cuối cùng, in ra summary để kiểm tra nhanh dataset đầu ra:

- Shape cuối cùng.
- Số sản phẩm unique.
- Số seller unique.
- Tỷ lệ dòng có review.
- Tỷ lệ dòng có product info.
- Top missing values sau xử lý.

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

main_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Đã lưu file tại:", OUTPUT_PATH)

print("\n===== FINAL SUMMARY =====")
print("Final shape:", main_df.shape)

if "asin" in main_df.columns:
    print("Số sản phẩm unique:", main_df["asin"].nunique())
else:
    print("Số sản phẩm unique: Không có cột asin")

if "productasin" in main_df.columns:
    print("Số productASIN unique:", main_df["productasin"].nunique())
else:
    print("Số productASIN unique: Không có cột productasin")

if "seller_name" in main_df.columns:
    print("Số seller unique:", main_df["seller_name"].nunique())
else:
    print("Số seller unique: Không có cột seller_name")

if "has_review" in main_df.columns:
    print(f"Tỷ lệ có review: {main_df['has_review'].mean():.2%}")

if "has_product_info" in main_df.columns:
    print(f"Tỷ lệ có product info: {main_df['has_product_info'].mean():.2%}")

print("\nTop 20 missing values sau xử lý:")
display(
    main_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Đã lưu file tại: c:\Users\ASPIRE 7\OneDrive\Tài liệu\UEH\CTD2026_DT049\Dataset\raw\amazon_products_reviews_merged.csv

===== FINAL SUMMARY =====
Final shape: (6327, 28)
Số sản phẩm unique: 700
Số productASIN unique: 700
Số seller unique: 260
Tỷ lệ có review: 99.40%
Tỷ lệ có product info: 98.34%

Top 20 missing values sau xử lý:


product_description    3987
manufacturer           3968
seller_page_url        2652
rank_1                 1442
best_sellers_rank      1349
recent_purchases       1084
productvariant          491
seller_name             105
price_value             105
availability             99
cleaned_review_text      39
reviewtext               38
review_text_clean        38
reviewmetadata           31
reviewtitle              14
rating                    7
sentiment_score           0
verifiedpurchase          0
product_text              0
has_review                0
dtype: int64